# PEARL — Benchmark CUDA del complesso MOL00583–EGFR
**Windows · RTX 4090 · kernel Python (PEARL GPU)**

Questo notebook misura il tempo di calcolo del sistema reale di 09h. Non prepara nuovi parametri e non richiede AmberTools/OpenFF: usa sistema, integratore e stato XML già esportati. Nessun file 09h viene sovrascritto. MOL00583 rimane un lead computazionale non validato sperimentalmente.

Il benchmark avanza una copia dello stato finale da 1 ns: **2 ps di riscaldamento computazionale** e **10 ps cronometrati**. Non è una nuova replica scientifica né un'analisi di convergenza. Non produce una traiettoria da usare nel report. I risultati servono a pianificare il calcolo.

1. Salvare questo notebook in `C:\Users\Roberto\PEARL`.
2. Selezionare **Python (PEARL GPU)**.
3. Eseguire **Kernel → Restart Kernel and Run All Cells**.
4. Inviare il riepilogo finale. La prima creazione del contesto può compilare i kernel CUDA e richiedere tempo: è esclusa dal throughput.

In [6]:
from pathlib import Path
import json, hashlib, time, sys, platform, statistics
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import openmm as mm
from openmm import app, unit

PEARL_ROOT = Path(r"C:\Users\Roberto\PEARL")
RUN_NAME = '20260912T174721_843455Z_seed20260912'
SOURCE = PEARL_ROOT/'outputs/pipeline_5_pharmacophore_09h_mol00583_md'/RUN_NAME
OUTPUT_BASE = PEARL_ROOT/'outputs/gpu_benchmark_MOL00583'
DEVICE_INDEX = '0'
PRECISION = 'mixed'
WARMUP_PS = 2.0
BLOCK_PS = 2.0
N_BLOCKS = 5
SEED = 20260914
MAC_REFERENCE_NS_DAY = 1.85 # throughput storico di 09h con reporter attivi; confronto indicativo
EXPECTED_SHA256 = {'states/production_system.xml': '512f12ff002b25a97a54a8bb78cf2c3527869011f8ba03360cfb6829357fc45d', 'states/production_integrator.xml': 'c8931563c5a025b070dedcabc366f468aa4a8f671bf317dcabf843b925a6ad59', 'states/production_final.xml': '223e5e76f341b87533769a76be9b05c8c06b3d34ef5cf9fd9fd2f380cdee639d', 'structures/solvated.pdb': '9b79a90eeea35fccb6381a728fa2bff3a5bf26edc5db67d6b35b9fe98828a006'}

def require(condition, message):
    if not condition: raise RuntimeError(message)
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def dump(path,obj):
    Path(path).write_text(json.dumps(obj,indent=2,default=str),encoding='utf-8')

print('Python del kernel:',sys.executable)
print('OpenMM:',mm.__version__)
require(PEARL_ROOT.is_dir(),f'Cartella PEARL assente: {PEARL_ROOT}')
for relative,expected in EXPECTED_SHA256.items():
    p=SOURCE/relative
    require(p.is_file(),f'File mancante: {p}')
    require(sha(p)==expected,f'Hash diverso: {p}. Verificare il trasferimento; non ignorare il controllo.')
print('Tutti e quattro i file sorgente sono integri.',flush=True)

Python del kernel: C:\Users\Roberto\anaconda3\envs\pearl-gpu\python.exe
OpenMM: 8.6.1
Tutti e quattro i file sorgente sono integri.


## 1. Caricamento e controllo del sistema
Si usa il sistema di produzione **senza restrizioni posizionali**, con barostato NPT. Le coordinate, velocità e dimensioni della scatola provengono dallo stato finale di 09h; nessun checkpoint binario del Mac viene usato. Il nuovo seme non trasforma questo breve benchmark in una replica indipendente.

In [8]:
system=mm.XmlSerializer.deserialize((SOURCE/'states/production_system.xml').read_text())
integrator=mm.XmlSerializer.deserialize((SOURCE/'states/production_integrator.xml').read_text())
state=mm.XmlSerializer.deserialize((SOURCE/'states/production_final.xml').read_text())
pdb=app.PDBFile(str(SOURCE/'structures/solvated.pdb'))
require(system.getNumParticles()==pdb.topology.getNumAtoms(),'Numero particelle/topologia incompatibile.')
require(len(state.getPositions())==system.getNumParticles(),'Stato con numero di particelle diverso.')
require(not any(isinstance(f,mm.CustomExternalForce) for f in system.getForces()),'Restrizioni esterne inattese.')
barostats=[f for f in system.getForces() if isinstance(f,mm.MonteCarloBarostat)]
require(len(barostats)==1 and barostats[0].getFrequency()>0,'Barostato NPT assente/non attivo.')
require(isinstance(integrator,mm.LangevinMiddleIntegrator),'Integratore inatteso.')
require(np.isfinite(state.getPositions(asNumpy=True).value_in_unit(unit.nanometer)).all(),'Coordinate non finite.')
require(np.isfinite(state.getVelocities(asNumpy=True).value_in_unit(unit.nanometer/unit.picosecond)).all(),'Velocita non finite.')
DT_PS=integrator.getStepSize().value_in_unit(unit.picosecond)
require(abs(DT_PS-0.002)<1e-10,'Il benchmark deve mantenere il timestep 09h di 2 fs.')
def steps(ps):
    n=round(ps/DT_PS)
    require(n>0 and abs(n*DT_PS-ps)<1e-8,'Durata non multipla del timestep.')
    return n
integrator.setRandomNumberSeed(SEED)
barostats[0].setRandomNumberSeed(SEED+1)
print('Particelle:',system.getNumParticles(),'| timestep:',DT_PS*1000,'fs')
print('Warm-up:',WARMUP_PS,'ps | misurazione:',N_BLOCKS*BLOCK_PS,'ps')

Particelle: 310408 | timestep: 2.0 fs
Warm-up: 2.0 ps | misurazione: 10.0 ps


## 2. Creazione del contesto CUDA
Nessun fallback a CPU: se CUDA non è disponibile il notebook si ferma. La proprietà `DeviceName` conferma il dispositivo realmente utilizzato. Questo passaggio può richiedere compilazione iniziale; non interromperlo soltanto perché non compaiono aggiornamenti immediati.

In [3]:
available=[mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
require('CUDA' in available,f'CUDA assente. Piattaforme: {available}; verificare il kernel pearl-gpu.')
RUN=OUTPUT_BASE/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN.mkdir(parents=True,exist_ok=False)
provenance={'status':'creating_context','source':str(SOURCE),'source_sha256':EXPECTED_SHA256,
 'OpenMM_version':mm.__version__,'python':sys.version,'host':platform.platform(),
 'precision':PRECISION,'device_index':DEVICE_INDEX,'seed':SEED,'dt_ps':DT_PS,
 'warmup_ps':WARMUP_PS,'block_ps':BLOCK_PS,'n_blocks':N_BLOCKS,'particle_count':system.getNumParticles(),
 'candidate':'MOL00583','experimentally_validated':False,'purpose':'performance_benchmark_only',
 'trajectory_reporters':False,'checkpoint_reporters':False}
dump(RUN/'provenance.json',provenance)
print('Creazione contesto CUDA...',flush=True)
start_context=time.perf_counter()
try:
    cuda=mm.Platform.getPlatformByName('CUDA')
    simulation=app.Simulation(pdb.topology,system,integrator,cuda,
                             {'DeviceIndex':DEVICE_INDEX,'Precision':PRECISION})
    # Set only physical state fields, avoiding platform-specific/context parameter mismatch.
    simulation.context.setPeriodicBoxVectors(*state.getPeriodicBoxVectors())
    simulation.context.setPositions(state.getPositions())
    simulation.context.setVelocities(state.getVelocities())
    simulation.context.setTime(0*unit.picosecond)
    simulation.currentStep=0
    device=cuda.getPropertyValue(simulation.context,'DeviceName')
    require('4090' in device,f'Dispositivo diverso dalla RTX 4090 attesa: {device}')
    provenance.update(status='ready',device_name=device,context_creation_seconds=time.perf_counter()-start_context)
    dump(RUN/'provenance.json',provenance)
    print('GPU effettiva:',device,'| precisione:',cuda.getPropertyValue(simulation.context,'Precision'),flush=True)
except Exception as exc:
    provenance.update(status='failed',error=repr(exc));dump(RUN/'provenance.json',provenance)
    raise

Creazione contesto CUDA...
GPU effettiva: NVIDIA GeForce RTX 4090 | precisione: mixed


## 3. Warm-up e benchmark con avanzamento
Ogni lettura di energia sincronizza il lavoro GPU prima di fermare il cronometro. Le scritture CSV e le stampe sono fuori dal tempo misurato. Non si scrivono DCD/checkpoint: il valore misura principalmente l'integrazione e può essere più ottimistico di una produzione con output completi.

In [4]:
def checked_energy():
    value=simulation.context.getState(getEnergy=True).getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
    require(np.isfinite(value),'Energia non finita: benchmark interrotto.')
    return float(value)
rows=[]
try:
    initial_energy=checked_energy()
    print('Warm-up GPU in corso...',flush=True)
    start=time.perf_counter();simulation.step(steps(WARMUP_PS));checked_energy()
    provenance['warmup_seconds']=time.perf_counter()-start
    provenance['status']='benchmark_running';dump(RUN/'provenance.json',provenance)
    for block in range(1,N_BLOCKS+1):
        start=time.perf_counter()
        simulation.step(steps(BLOCK_PS))
        energy=checked_energy()
        elapsed=time.perf_counter()-start
        speed=(BLOCK_PS/1000)*86400/elapsed
        rows.append({'block':block,'simulated_ps':BLOCK_PS,'seconds':elapsed,'ns_per_day':speed,
                     'potential_energy_kJ_mol':energy})
        pd.DataFrame(rows).to_csv(RUN/'benchmark_blocks.csv',index=False)
        eta=sum(r['seconds'] for r in rows)/len(rows)*(N_BLOCKS-block)
        print(f'{block}/{N_BLOCKS} ({100*block/N_BLOCKS:.0f}%) | {speed:.2f} ns/giorno | '
              f'blocco: {elapsed:.1f} s | restano circa {eta:.0f} s',flush=True)
    final_state=simulation.context.getState(getPositions=True,getEnergy=True)
    require(np.isfinite(final_state.getPositions(asNumpy=True).value_in_unit(unit.nanometer)).all(),
            'Coordinate finali non finite.')
except Exception as exc:
    provenance.update(status='failed',error=repr(exc),completed_blocks=len(rows));dump(RUN/'provenance.json',provenance)
    raise
finally:
    # Release GPU memory after this short benchmark; no scientific continuation is saved.
    if 'simulation' in globals(): del simulation

Warm-up GPU in corso...
1/5 (20%) | 100.37 ns/giorno | blocco: 1.7 s | restano circa 7 s
2/5 (40%) | 101.70 ns/giorno | blocco: 1.7 s | restano circa 5 s
3/5 (60%) | 101.79 ns/giorno | blocco: 1.7 s | restano circa 3 s
4/5 (80%) | 104.92 ns/giorno | blocco: 1.6 s | restano circa 2 s
5/5 (100%) | 105.72 ns/giorno | blocco: 1.6 s | restano circa 0 s


## 4. Risultati e stime per pianificare il lavoro
Le stime riguardano sola produzione, assumono velocità costante e repliche eseguite una alla volta. Escludono preparazione, equilibrazione, analisi, output, pause e altri carichi della GPU. Le durate sono scenari di costo, non una raccomandazione di durata scientifica.

Il confronto con 1,85 ns/giorno del Mac è indicativo: il run Mac scriveva traiettorie/checkpoint, questo benchmark no. Il throughput misurato per il complesso non si trasferisce automaticamente al ligando libero in acqua o a un protocollo alchemico.

In [5]:
blocks=pd.DataFrame(rows)
require(len(blocks)==N_BLOCKS,'Benchmark incompleto.')
total_ns=blocks.simulated_ps.sum()/1000
speed=total_ns*86400/blocks.seconds.sum()
cv=float(blocks.ns_per_day.std(ddof=1)/blocks.ns_per_day.mean()) if len(blocks)>1 else 0.0
scenarios=pd.DataFrame([
 {'scenario':'1 ns','total_ns':1},
 {'scenario':'10 ns','total_ns':10},
 {'scenario':'3 repliche da 10 ns (sequenziali)','total_ns':30},
 {'scenario':'50 ns','total_ns':50}])
scenarios['hours_estimated']=scenarios.total_ns/speed*24
scenarios['days_estimated']=scenarios.total_ns/speed
scenarios.to_csv(RUN/'runtime_estimates.csv',index=False)
result={'status':'completed','device':device,'OpenMM':mm.__version__,'precision':PRECISION,
 'particles':system.getNumParticles(),'measured_ps':float(blocks.simulated_ps.sum()),
 'ns_per_day':float(speed),'speed_min_block':float(blocks.ns_per_day.min()),
 'speed_max_block':float(blocks.ns_per_day.max()),'relative_block_SD':cv,
 'speedup_vs_Mac_indicative':float(speed/MAC_REFERENCE_NS_DAY),
 'technical_qc_pass':True,'timing_variability_flag':cv>0.15,
 'note':'Benchmark breve senza DCD/checkpoint. Stime di sola produzione, non risultati scientifici.'}
dump(RUN/'benchmark_summary.json',result)
provenance.update(status='completed',result=result);dump(RUN/'provenance.json',provenance)
print(json.dumps(result,indent=2))
display(scenarios)
if cv>0.15:print('Velocita variabile: ripetere con blocchi piu lunghi e GPU libera prima di pianificare.')
print('Risultati salvati in:',RUN)
print('Inviare benchmark_summary.json oppure il notebook salvato con gli output.')

{
  "status": "completed",
  "device": "NVIDIA GeForce RTX 4090",
  "OpenMM": "8.6.1",
  "precision": "mixed",
  "particles": 310408,
  "measured_ps": 10.0,
  "ns_per_day": 102.85871022796587,
  "speed_min_block": 100.36551517567572,
  "speed_max_block": 105.72207257844023,
  "relative_block_SD": 0.022343063431359864,
  "speedup_vs_Mac_indicative": 55.599302825927495,
  "technical_qc_pass": true,
  "timing_variability_flag": false,
  "note": "Benchmark breve senza DCD/checkpoint. Stime di sola produzione, non risultati scientifici."
}


,scenario,total_ns,hours_estimated,days_estimated
0,1 ns,1,0.233330,0.009722
1,10 ns,10,2.333298,0.097221
2,3 repliche da 10 ns (sequenziali),30,6.999893,0.291662
3,50 ns,50,11.666489,0.486104


Risultati salvati in: C:\Users\Roberto\PEARL\outputs\gpu_benchmark_MOL00583\20260914T072148_859898Z
Inviare benchmark_summary.json oppure il notebook salvato con gli output.
